# Probability Integral Transform (PIT) Analysis



In [ ]:
import sys

sys.path.append("../../src/")

import pickle

import matplotlib as mpl
import matplotlib.pyplot as plt
from cycler import cycler

import numpy as np
import pandas as pd
from lightning import seed_everything
from scipy import stats

from model_evaluation.utils import compute_crps, compute_mpiw, compute_picp
from model_training.data_modules.utils import EPFDataModule

pgf_backend = True
if pgf_backend:
    mpl.use("pgf")

In [ ]:
# IEEE Access matplotlib settings
# Source: https://journals.ieeeauthorcenter.ieee.org/create-your-ieee-journal-article/create-graphics-for-your-article/resolution-and-size/

font_size = 8

markers = [
    "o",
    "s",
    "^",
    "D",
    "v",
    "P",
    "X",
    "*",
    "<",
    ">",
    "p",
    "h",
    "H",
    "d",
    "|",
    "_",
    "1",
    "2",
    "3",
    "4",
    "+",
    "x",
    ".",
    ",",
]
colors = list(mpl.colormaps["tab10"].colors) * ((len(markers) // 10) + 1)
colors = colors[: len(markers)]

plt.rcParams.update(
    {
        "pgf.rcfonts": False,  # don't use matplotlib defaults
        "pgf.texsystem": "lualatex",  # use LuaLaTeX
        "font.family": "serif",
        "font.serif": ["Times New Roman"],  # system Times New Roman
        "figure.dpi": 300,
        "font.size": font_size,
        "axes.titlesize": font_size,
        "figure.titlesize": font_size,
        "axes.labelsize": font_size,
        "legend.fontsize": font_size,
        "xtick.labelsize": font_size,
        "ytick.labelsize": font_size,
        "pgf.preamble": r"\usepackage{fontspec}\setmainfont{Times New Roman}",
        "axes.prop_cycle": cycler(color=colors)
        + cycler(marker=markers)
        + cycler(markevery=[10] * len(markers)),
        "lines.markersize": 3,
        "lines.markeredgewidth": 0.0,
    }
)

# IEEE standard widths (inches)
linewidth_singlecol = 3.5  # single-column figure
linewidth_doublecol = 7.16  # double-column figure
max_height = 9.25  # max text height

golden_ratio = (5**0.5 - 1) / 2

dpi = 300


def set_size(width=linewidth_singlecol, ratio=golden_ratio, height_pad=0):
    height = width * ratio + height_pad
    return (width, min(height, max_height))

In [ ]:
seed_everything(0)

In [ ]:
n_runs = 10
standardization_case = "mean_std"

## Load Data

In [ ]:
# Load the data
val_date = "2022-12-01"
test_date = "2023-12-01"
end_date = "2024-11-30"
data_file_path = "../../data/processed/smard_data_201810010000_202501010000.npz"
data_module = EPFDataModule(
    data_file_path=data_file_path,
    val_date=val_date,
    test_date=test_date,
    end_date=end_date,
    batch_size=32,
    standardization_case=standardization_case,
)

train_input, train_labels = data_module.train_dataset[:]
val_input, val_labels = data_module.val_dataset[:]
test_input, test_labels = data_module.test_dataset[:]

data_input, data_labels = test_input, test_labels

data_labels = data_labels * data_module.scale_target + data_module.offset_target
data_labels = data_labels.detach().numpy().astype(np.float64)

In [ ]:
quantiles = np.linspace(0.01, 0.99, 99)

## Load Models

In [ ]:
models_dict = []

In [ ]:
# Load your models here
file_paths = [
    "../evaluate_models/results/metric_evaluation/NaiveHS.pkl",
    "../evaluate_models/results/metric_evaluation/DDNN_Ens.pkl",
    "../evaluate_models/results/metric_evaluation/MCD.pkl",
    "../evaluate_models/results/metric_evaluation/EvDNN.pkl",
    "../evaluate_models/results/metric_evaluation/DDNN_CP.pkl",
    "../evaluate_models/results/metric_evaluation/Ens_CP.pkl",
    "../evaluate_models/results/metric_evaluation/MCD_CP.pkl",
    "../evaluate_models/results/metric_evaluation/EvDNN_CP.pkl",
    "../evaluate_models/results/metric_evaluation/LEAR_GARCH.pkl",
    "../evaluate_models/results/metric_evaluation/LEAR_QRA.pkl",
    "../evaluate_models/results/metric_evaluation/LEAR_CP.pkl",
    "../evaluate_models/results/metric_evaluation/XGBoost_GARCH.pkl",
    "../evaluate_models/results/metric_evaluation/XGBoost_QRA.pkl",
    "../evaluate_models/results/metric_evaluation/XGBoost_CP.pkl",
]

for file_path in file_paths:
    with open(file_path, "rb") as f:
        models_dict.extend(pickle.load(f))

In [ ]:
for model in models_dict:
    if model["model_name"] == "ddnn_normal":
        model["model_name"] = "DDNN"
    elif model["model_name"] == "ens5_normal":
        model["model_name"] = "Ens5"
    elif model["model_name"] == "ens10_normal":
        model["model_name"] = "Ens10"
    elif model["model_name"] == "mcd10_normal":
        model["model_name"] = "MCD10"
    elif model["model_name"] == "mcd30_normal":
        model["model_name"] = "MCD30"
    elif model["model_name"] == "naive_hs_train_normal":
        model["model_name"] = "Naive-HS$_{train}$"
    elif model["model_name"] == "naive_hs_val_normal":
        model["model_name"] = "Naive-HS$_{val}$"
    elif model["model_name"] == "evdnn_normal":
        model["model_name"] = "EvDNN"
    elif model["model_name"] == "lasso_garch":
        model["model_name"] = "LASSO-GARCH"
    elif model["model_name"] == "LEAR_GARCH":
        model["model_name"] = "LEAR-GARCH"
    elif model["model_name"] == "LEAR_QRA":
        model["model_name"] = "LEAR-QRA"
    elif model["model_name"] == "XGBoost_GARCH":
        model["model_name"] = "XGBoost-GARCH"
    elif model["model_name"] == "XGBoost_QRA":
        model["model_name"] = "XGBoost-QRA"

models_to_exclude = {
    "Ens5",
    "MCD10",
    "Naive-HS$_{train}$",
    "EvDNN",
    "EvDNN-CP",
    "MCD30-CP",
    "Ens10-CP",
}
models_dict = [
    model for model in models_dict if model["model_name"] not in models_to_exclude
]

In [ ]:
print(f"Loaded {len(models_dict)} models")
print(f"Model names: {[m['model_name'] for m in models_dict]}")
print(f"Quantile shape: {models_dict[0]['quantile'].shape}")
print(f"Data labels shape: {data_labels.shape}")
print(f"Number of days: {data_labels.shape[0]}, Number of hours: {data_labels.shape[1]}")

## Compute PIT Values

In [ ]:
def compute_pit(quantiles, model_quantile, data_labels,*, interpolate=False):
    """
    Calculate Probability Integral Transform (PIT) values.

    Args:
        quantiles: Array of quantile levels (e.g., 0.01 to 0.99), shape (n_quantiles,)
        model_quantile: Predicted quantiles, shape (n_runs, n_quantiles, n_days, n_hours)
        data_labels: True values, shape (n_days, n_hours)
        interpolate: Whether to use interpolation for PIT calculation

    Returns:
        PIT values with shape (n_runs, n_days, n_hours)
    """
    n_runs, _, n_days, n_hours = model_quantile.shape
    pit_values = np.zeros((n_runs, n_days, n_hours))

    for run in range(n_runs):
        for day in range(n_days):
            for hour in range(n_hours):
                if interpolate:
                # Find the CDF value at the observed point by interpolation
                    pit_values[run, day, hour] = np.interp(
                        data_labels[day, hour],
                        model_quantile[run, :, day, hour],
                        quantiles,
                    )
                else:
                    raise NotImplementedError("Non-interpolated PIT calculation is not implemented.")
    return pit_values

In [ ]:
# Compute PIT for all models
for model in models_dict:
    print(f"Computing PIT for {model['model_name']}...")
    model["pit"] = compute_pit(quantiles, model["quantile"], data_labels, interpolate=True)
    model["pit_mean"] = np.mean(model["pit"], axis=0)  # Average over runs
    model["pit_flat"] = model["pit"].flatten()

    # Compute PIT by hour of day
    model["pit_by_hour"] = {}
    for hour in range(24):
        model["pit_by_hour"][hour] = model["pit"][:, :, hour].flatten()

print("PIT computation complete!")

## Overall PIT Histograms

In [ ]:
# Plot overall PIT histograms for all models
n_models = len(models_dict)
n_cols = min(3, n_models)
n_rows = (n_models + n_cols - 1) // n_cols

# fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
figsize = set_size(width=linewidth_doublecol, height_pad=3)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=figsize #, sharex=True #, gridspec_kw={"height_ratios": [1, 1]}
)
if n_models == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, model in enumerate(models_dict):
    ax = axes[idx]

    # Plot histogram
    ax.hist(
        model["pit_flat"],
        bins=20,
        density=True,
        alpha=0.7,
        edgecolor="black",
        color="steelblue",
    )

    # Add uniform distribution reference line
    ax.axhline(y=1.0, color="red", linestyle="--", linewidth=1, label="Uniform")

    ax.set_title(model["model_name"], fontweight="bold")#, fontsize=24)
    ax.set_xlabel("PIT values")#, fontsize=20)
    ax.set_ylabel("Density")#, fontsize=20)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 2.5)
    # ax.tick_params(axis='both', which='major', labelsize=20)
    ax.legend()#fontsize=20)
    ax.grid(alpha=0.3)

# Hide extra subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig("results/pit_histograms_overall.pdf", dpi=300, bbox_inches="tight")
plt.show()

## PIT Statistics Summary

In [ ]:
# Compute summary statistics for PIT values
summary_data = []

for model in models_dict:

    summary_data.append(
        {
            "Model": model["model_name"],
            "Mean": np.mean(model["pit_flat"]),
            "Std": np.std(model["pit_flat"]),
            "Skewness": stats.skew(model["pit_flat"]),
            "Kurtosis": stats.kurtosis(model["pit_flat"]),
        }
    )

summary_df = pd.DataFrame(summary_data)
summary_df

## Q-Q Plot for Uniformity Check

In [ ]:
# Q-Q plot to visually assess uniformity
n_models = len(models_dict)
n_cols = min(3, n_models)
n_rows = (n_models + n_cols - 1) // n_cols

# fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
figsize = set_size(width=linewidth_doublecol, height_pad=3)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=figsize #, sharex=True #, gridspec_kw={"height_ratios": [1, 1]}
)
if n_models == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, model in enumerate(models_dict):
    ax = axes[idx]

    # Sort PIT values
    sorted_pit = np.sort(model["pit_flat"])
    theoretical_quantiles = np.linspace(0, 1, len(sorted_pit))

    # Q-Q plot
    ax.plot([0, 1], [0, 1], "r--", linewidth=1, label="Perfect calibration")
    ax.scatter(theoretical_quantiles, sorted_pit, alpha=0.3, s=0.5, rasterized=True)
    ax.set_title(model["model_name"], fontweight="bold")
    ax.set_xlabel("Theoretical Quantiles")#, fontsize=20)
    ax.set_ylabel("Sample Quantiles")#, fontsize=20)
    ax.legend()#fontsize=20)
    # ax.tick_params(axis='both', which='major', labelsize=20)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

# Hide extra subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig("results/pit_qq_plots.pdf", dpi=300, bbox_inches="tight")
plt.show()